## Cell 1 — Import library

In [13]:
import pandas as pd
from pathlib import Path

## Cell 2 — Tentukan folder project

In [14]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Output tables dir:", OUTPUT_TABLES_DIR)

Project root: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL
Processed dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed
Output tables dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables


## Cell 3 — Baca event log process-ready

In [15]:
EVENT_LOG_PROCESS_READY_PATH = PROCESSED_DIR / "03_event_log_process_ready.csv"

event_log_full = pd.read_csv(
    EVENT_LOG_PROCESS_READY_PATH,
    parse_dates=["timestamp"]
)

print("Ukuran event_log_full:", event_log_full.shape)
display(event_log_full.head())

print("\nJumlah mahasiswa:", event_log_full["case_id"].nunique())
print("Jumlah activity_mapped unik:", event_log_full["activity_mapped"].nunique())
print("Timestamp awal:", event_log_full["timestamp"].min())
print("Timestamp akhir:", event_log_full["timestamp"].max())

Ukuran event_log_full: (147249, 16)


,case_id,activity,timestamp,kelas,Component,Event context,Description,Origin,IP address,nim,nama,nilai_total,nilai_indeks,label_performance,activity_mapped,activity_category
0,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 13:30:09,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.204,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Course Viewed,course
1,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 13:31:17,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.203,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Course Viewed,course
2,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 13:39:55,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Course Viewed,course
3,ADITYA PUTRA PERMANA,Course module viewed,2025-09-18 14:24:42,IF-49-02,File,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Material Viewed,material
4,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 15:41:24,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Course Viewed,course



Jumlah mahasiswa: 89
Jumlah activity_mapped unik: 25
Timestamp awal: 2025-09-16 09:49:33
Timestamp akhir: 2026-01-16 22:18:34


## Cell 4 — Pastikan urutan event benar

In [16]:
event_log_full = event_log_full.sort_values(
    by=["case_id", "timestamp"]
).reset_index(drop=True)

event_log_full["event_order"] = (
    event_log_full
    .groupby("case_id")
    .cumcount() + 1
)

display(event_log_full[["case_id", "event_order", "activity_mapped", "timestamp"]].head(20))

,case_id,event_order,activity_mapped,timestamp
0,ADITYA PUTRA PERMANA,1,Course Viewed,2025-09-18 13:30:09
1,ADITYA PUTRA PERMANA,2,Course Viewed,2025-09-18 13:31:17
2,ADITYA PUTRA PERMANA,3,Course Viewed,2025-09-18 13:39:55
3,ADITYA PUTRA PERMANA,4,Material Viewed,2025-09-18 14:24:42
4,ADITYA PUTRA PERMANA,5,Course Viewed,2025-09-18 15:41:24
5,ADITYA PUTRA PERMANA,6,Material Viewed,2025-09-18 15:41:57
6,ADITYA PUTRA PERMANA,7,Course Viewed,2025-09-18 16:40:55
7,ADITYA PUTRA PERMANA,8,Course Viewed,2025-09-18 16:49:33
8,ADITYA PUTRA PERMANA,9,Course Viewed,2025-09-18 16:49:56
9,ADITYA PUTRA PERMANA,10,Quiz Module Viewed,2025-09-18 16:50:56


## Cell 5 — Buat compact log

In [17]:
event_log_full["previous_activity"] = (
    event_log_full
    .groupby("case_id")["activity_mapped"]
    .shift(1)
)

event_log_compact = event_log_full[
    event_log_full["activity_mapped"] != event_log_full["previous_activity"]
].copy()

event_log_compact = event_log_compact.drop(columns=["previous_activity"])
event_log_full = event_log_full.drop(columns=["previous_activity"])

event_log_compact = event_log_compact.sort_values(
    by=["case_id", "timestamp"]
).reset_index(drop=True)

event_log_compact["compact_event_order"] = (
    event_log_compact
    .groupby("case_id")
    .cumcount() + 1
)

print("Ukuran full log:", event_log_full.shape)
print("Ukuran compact log:", event_log_compact.shape)

display(event_log_compact[["case_id", "compact_event_order", "activity_mapped", "timestamp"]].head(20))

Ukuran full log: (147249, 17)
Ukuran compact log: (116611, 18)


,case_id,compact_event_order,activity_mapped,timestamp
0,ADITYA PUTRA PERMANA,1,Course Viewed,2025-09-18 13:30:09
1,ADITYA PUTRA PERMANA,2,Material Viewed,2025-09-18 14:24:42
2,ADITYA PUTRA PERMANA,3,Course Viewed,2025-09-18 15:41:24
3,ADITYA PUTRA PERMANA,4,Material Viewed,2025-09-18 15:41:57
4,ADITYA PUTRA PERMANA,5,Course Viewed,2025-09-18 16:40:55
5,ADITYA PUTRA PERMANA,6,Quiz Module Viewed,2025-09-18 16:50:56
6,ADITYA PUTRA PERMANA,7,Course Viewed,2025-09-18 16:51:17
7,ADITYA PUTRA PERMANA,8,Material Viewed,2025-09-18 17:36:04
8,ADITYA PUTRA PERMANA,9,Course Viewed,2025-09-18 20:36:54
9,ADITYA PUTRA PERMANA,10,Material Viewed,2025-09-18 20:37:38


## Cell 6 — Ringkasan full log vs compact log

In [18]:
summary_full_vs_compact = pd.DataFrame({
    "jenis_log": ["Full log", "Compact log"],
    "jumlah_event": [
        len(event_log_full),
        len(event_log_compact)
    ],
    "jumlah_mahasiswa": [
        event_log_full["case_id"].nunique(),
        event_log_compact["case_id"].nunique()
    ],
    "jumlah_activity_unik": [
        event_log_full["activity_mapped"].nunique(),
        event_log_compact["activity_mapped"].nunique()
    ],
    "rata_rata_event_per_mahasiswa": [
        len(event_log_full) / event_log_full["case_id"].nunique(),
        len(event_log_compact) / event_log_compact["case_id"].nunique()
    ]
})

summary_full_vs_compact["rata_rata_event_per_mahasiswa"] = (
    summary_full_vs_compact["rata_rata_event_per_mahasiswa"].round(2)
)

reduction_percentage = (
    (1 - len(event_log_compact) / len(event_log_full)) * 100
)

display(summary_full_vs_compact)

print("Persentase pengurangan event setelah compact log:", round(reduction_percentage, 2), "%")

,jenis_log,jumlah_event,jumlah_mahasiswa,jumlah_activity_unik,rata_rata_event_per_mahasiswa
0,Full log,147249,89,25,1654.48
1,Compact log,116611,89,25,1310.24


Persentase pengurangan event setelah compact log: 20.81 %


## Cell 7 — Distribusi aktivitas full log

In [19]:
activity_full_counts = (
    event_log_full["activity_mapped"]
    .value_counts()
    .reset_index()
)

activity_full_counts.columns = ["activity_mapped", "jumlah_event_full"]
activity_full_counts["persentase_full"] = (
    activity_full_counts["jumlah_event_full"] / len(event_log_full) * 100
).round(2)

display(activity_full_counts)

,activity_mapped,jumlah_event_full,persentase_full
0,Quiz Viewed,50802,34.50
1,Quiz Updated,49343,33.51
2,Quiz Module Viewed,10536,7.16
3,Course Viewed,10456,7.10
4,Material Viewed,7343,4.99
5,Quiz Auto-saved,5365,3.64
6,Quiz Access Prevented,3521,2.39
7,Quiz Summary Viewed,2392,1.62
8,Quiz Started,2259,1.53
9,Quiz Submitted,2210,1.50


## Cell 8 — Distribusi aktivitas compact log

In [20]:
activity_compact_counts = (
    event_log_compact["activity_mapped"]
    .value_counts()
    .reset_index()
)

activity_compact_counts.columns = ["activity_mapped", "jumlah_event_compact"]
activity_compact_counts["persentase_compact"] = (
    activity_compact_counts["jumlah_event_compact"] / len(event_log_compact) * 100
).round(2)

display(activity_compact_counts)

,activity_mapped,jumlah_event_compact,persentase_compact
0,Quiz Viewed,42249,36.23
1,Quiz Updated,41209,35.34
2,Quiz Module Viewed,7348,6.30
3,Course Viewed,6384,5.47
4,Quiz Auto-saved,4594,3.94
5,Material Viewed,3405,2.92
6,Quiz Summary Viewed,2362,2.03
7,Quiz Started,2259,1.94
8,Quiz Submitted,2210,1.90
9,Quiz Access Prevented,1783,1.53


## Cell 9 — Bandingkan aktivitas full vs compact

In [21]:
activity_comparison = activity_full_counts.merge(
    activity_compact_counts,
    on="activity_mapped",
    how="outer"
).fillna(0)

activity_comparison["pengurangan_event"] = (
    activity_comparison["jumlah_event_full"] - activity_comparison["jumlah_event_compact"]
)

activity_comparison["persentase_pengurangan"] = (
    activity_comparison["pengurangan_event"] / activity_comparison["jumlah_event_full"] * 100
).round(2)

activity_comparison = activity_comparison.sort_values(
    "jumlah_event_full",
    ascending=False
)

display(activity_comparison)

,activity_mapped,jumlah_event_full,persentase_full,jumlah_event_compact,persentase_compact,pengurangan_event,persentase_pengurangan
24,Quiz Viewed,50802,34.50,42249,36.23,8553,16.84
23,Quiz Updated,49343,33.51,41209,35.34,8134,16.48
19,Quiz Module Viewed,10536,7.16,7348,6.30,3188,30.26
12,Course Viewed,10456,7.10,6384,5.47,4072,38.94
16,Material Viewed,7343,4.99,3405,2.92,3938,53.63
18,Quiz Auto-saved,5365,3.64,4594,3.94,771,14.37
17,Quiz Access Prevented,3521,2.39,1783,1.53,1738,49.36
22,Quiz Summary Viewed,2392,1.62,2362,2.03,30,1.25
20,Quiz Started,2259,1.53,2259,1.94,0,0.00
21,Quiz Submitted,2210,1.50,2210,1.90,0,0.00


## Cell 10 — Fungsi membuat transisi directly-follows

In [22]:
def create_transition_table(df, activity_col="activity_mapped"):
    temp = df.sort_values(["case_id", "timestamp"]).copy()

    temp["next_activity"] = (
        temp
        .groupby("case_id")[activity_col]
        .shift(-1)
    )

    transitions = temp.dropna(subset=["next_activity"]).copy()

    transition_counts = (
        transitions
        .groupby([activity_col, "next_activity"])
        .size()
        .reset_index(name="frequency")
        .sort_values("frequency", ascending=False)
    )

    transition_counts = transition_counts.rename(columns={
        activity_col: "source",
        "next_activity": "target"
    })

    return transition_counts

## Cell 11 — Buat transisi full log dan compact log

In [23]:
transitions_full = create_transition_table(event_log_full)
transitions_compact = create_transition_table(event_log_compact)

print("Jumlah jenis transisi full log:", len(transitions_full))
print("Jumlah jenis transisi compact log:", len(transitions_compact))

print("\nTop 20 transisi full log:")
display(transitions_full.head(20))

print("\nTop 20 transisi compact log:")
display(transitions_compact.head(20))

Jumlah jenis transisi full log: 177
Jumlah jenis transisi compact log: 164

Top 20 transisi full log:


,source,target,frequency
175,Quiz Viewed,Quiz Updated,38709
164,Quiz Updated,Quiz Viewed,35519
176,Quiz Viewed,Quiz Viewed,8553
163,Quiz Updated,Quiz Updated,8134
64,Course Viewed,Course Viewed,4072
90,Material Viewed,Material Viewed,3938
123,Quiz Module Viewed,Quiz Module Viewed,3188
118,Quiz Module Viewed,Course Viewed,2990
159,Quiz Updated,Quiz Auto-saved,2864
114,Quiz Auto-saved,Quiz Viewed,2853



Top 20 transisi compact log:


,source,target,frequency
163,Quiz Viewed,Quiz Updated,38709
152,Quiz Updated,Quiz Viewed,35519
109,Quiz Module Viewed,Course Viewed,2990
148,Quiz Updated,Quiz Auto-saved,2864
105,Quiz Auto-saved,Quiz Viewed,2853
61,Course Viewed,Material Viewed,2824
82,Material Viewed,Course Viewed,2434
64,Course Viewed,Quiz Module Viewed,2228
151,Quiz Updated,Quiz Summary Viewed,2180
117,Quiz Module Viewed,Quiz Viewed,1991


## Cell 12 — Simpan full log, compact log, dan tabel

In [24]:
FULL_LOG_OUTPUT_PATH = PROCESSED_DIR / "04_full_log_process_ready.csv"
COMPACT_LOG_OUTPUT_PATH = PROCESSED_DIR / "04_compact_log_process_ready.csv"

event_log_full.to_csv(FULL_LOG_OUTPUT_PATH, index=False)
event_log_compact.to_csv(COMPACT_LOG_OUTPUT_PATH, index=False)

summary_full_vs_compact.to_csv(
    OUTPUT_TABLES_DIR / "04_summary_full_vs_compact.csv",
    index=False
)

activity_comparison.to_csv(
    OUTPUT_TABLES_DIR / "04_activity_full_vs_compact_comparison.csv",
    index=False
)

transitions_full.to_csv(
    OUTPUT_TABLES_DIR / "04_transitions_full_log.csv",
    index=False
)

transitions_compact.to_csv(
    OUTPUT_TABLES_DIR / "04_transitions_compact_log.csv",
    index=False
)

print("Full log disimpan ke:")
print(FULL_LOG_OUTPUT_PATH)

print("\nCompact log disimpan ke:")
print(COMPACT_LOG_OUTPUT_PATH)

print("\nTabel hasil Notebook 04 disimpan ke:")
print(OUTPUT_TABLES_DIR)

Full log disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\04_full_log_process_ready.csv

Compact log disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\04_compact_log_process_ready.csv

Tabel hasil Notebook 04 disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables
